# Chapter 11 — Performance

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 11.

---


### Item 92 — Profile Before Optimizing

Intuition about which part of a program is "the slow part" is frequently wrong — the standard library's `cProfile` module measures where time is *actually* spent, function by function, so optimization effort goes to the real bottleneck instead of a guess. Optimizing before profiling routinely wastes effort on code that was never the actual problem.


In [1]:
import cProfile
import pstats
import io

def slow_part():
    return sum(i * i for i in range(200_000))

def fast_part():
    return sum(range(100))

def main():
    for _ in range(5):
        fast_part()
    slow_part()

profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumulative')
stats.print_stats(4)
print(stream.getvalue())


         200052 function calls in 0.041 seconds

   Ordered by: cumulative time
   List reduced from 22 to 4 due to restriction <4>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.041    0.020 /sessions/festive-peaceful-johnson/.local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3543(run_code)
        2    0.000    0.000    0.041    0.020 {built-in method builtins.exec}
        1    0.000    0.000    0.041    0.041 /sessions/festive-peaceful-johnson/tmp/ipykernel_10/2308831819.py:1(<module>)
        1    0.000    0.000    0.041    0.041 /sessions/festive-peaceful-johnson/tmp/ipykernel_10/2308831819.py:11(main)





### Item 93 — Optimize Performance-Critical Code Using `timeit` Microbenchmarks

Once profiling (Item 92) identifies a real hotspot, `timeit` measures a small snippet's execution time precisely — running it many times and controlling for noise — which is exactly what you need to confirm a proposed optimization is actually faster, rather than assuming it is.


In [2]:
import timeit

# Compare two ways of building a list of squares
list_comp_time = timeit.timeit('[x**2 for x in range(1000)]', number=1000)
map_time = timeit.timeit('list(map(lambda x: x**2, range(1000)))', number=1000)

print(f'list comprehension: {list_comp_time:.4f}s for 1000 runs')
print(f'map + lambda:       {map_time:.4f}s for 1000 runs')
print('timeit isolates the snippet and repeats it enough times to get a stable measurement.')


list comprehension: 0.1414s for 1000 runs
map + lambda:       0.1521s for 1000 runs
timeit isolates the snippet and repeats it enough times to get a stable measurement.


### Item 94 — Know When and How to Replace Python with Another Programming Language

For code that stays slow even after profiling and optimizing pure Python, and where a C-extension is overkill, sometimes the pragmatic move is rewriting just that hot piece in a faster language and calling it from Python (via `subprocess`, `ctypes`, or a proper extension module) — the book frames this as a legitimate, deliberate escape hatch for a *specific, proven* bottleneck, not a default strategy.


In [3]:
import subprocess, time

# A CPU-heavy task in pure Python
def python_sum_of_squares(n):
    return sum(i * i for i in range(n))

start = time.time()
result_py = python_sum_of_squares(5_000_000)
py_time = time.time() - start
print(f'pure Python: {result_py} in {py_time:.3f}s')

# The same computation done by a compiled C program, called via subprocess
c_code = '''
#include <stdio.h>
int main() {
    long long total = 0;
    for (long long i = 0; i < 5000000; i++) total += i * i;
    printf("%lld\\n", total);
    return 0;
}
'''
with open('sumsq.c', 'w') as f:
    f.write(c_code)
subprocess.run(['gcc', '-O2', 'sumsq.c', '-o', 'sumsq'], check=True)

start = time.time()
result_c = subprocess.run(['./sumsq'], capture_output=True, text=True).stdout.strip()
c_time = time.time() - start
print(f'compiled C via subprocess: {result_c} in {c_time:.3f}s')

import os
os.remove('sumsq.c')
os.remove('sumsq')


pure Python: 41666654166667500000 in 0.180s


compiled C via subprocess: 4773166019248396768 in 0.008s


### Item 95 — Consider `ctypes` to Rapidly Integrate with Native Libraries

`ctypes` (standard library) lets Python call functions in existing compiled shared libraries (`.so`/`.dll`) directly, with no compilation step of your own required — useful for quickly reaching functionality in a C library (or the system's own `libc`) without writing a full C extension module (Item 96) just to wrap one or two functions.


In [4]:
import ctypes
import ctypes.util

libc = ctypes.CDLL(None)   # loads the already-present C standard library, no compilation needed

libc.printf(b'Hello from libc, called directly via ctypes: %d\n', ctypes.c_int(42))

# ctypes also lets you declare argument/return types explicitly for safety
libm = ctypes.CDLL('libm.so.6') if ctypes.util.find_library('m') else ctypes.CDLL(None)
libm.sqrt.restype = ctypes.c_double
libm.sqrt.argtypes = [ctypes.c_double]
print('sqrt(2.0) via libm through ctypes:', libm.sqrt(2.0))


sqrt(2.0) via libm through ctypes: 1.4142135623730951


### Item 96 — Consider Extension Modules to Maximize Performance and Ergonomics

For code called *very* frequently where even ctypes overhead matters, a real C extension module (built with the Python C-API, via `setuptools`) compiles directly into a native Python module — imported and called exactly like pure-Python code, but running as compiled machine code with no per-call interpretation overhead. This example actually compiles a tiny real `fastmath` extension from C source and imports it.


In [5]:
import subprocess, sys, pathlib

work = pathlib.Path('ext_demo')
work.mkdir(exist_ok=True)

(work / 'fastmath.c').write_text('''
#define PY_SSIZE_T_CLEAN
#include <Python.h>

static PyObject* fastmath_square(PyObject* self, PyObject* args) {
    long n;
    if (!PyArg_ParseTuple(args, "l", &n)) return NULL;
    return PyLong_FromLong(n * n);
}

static PyMethodDef FastMathMethods[] = {
    {"square", fastmath_square, METH_VARARGS, "Compute the square of an integer."},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef fastmathmodule = {
    PyModuleDef_HEAD_INIT, "fastmath", NULL, -1, FastMathMethods
};

PyMODINIT_FUNC PyInit_fastmath(void) {
    return PyModule_Create(&fastmathmodule);
}
''')

(work / 'setup.py').write_text('''
from setuptools import setup, Extension
setup(name="fastmath", ext_modules=[Extension("fastmath", ["fastmath.c"])])
''')

build = subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd=work, capture_output=True, text=True
)
print('build exit code:', build.returncode)

sys.path.insert(0, str(work))
import fastmath   # a REAL compiled C extension, imported like any Python module
print('fastmath.square(12) ->', fastmath.square(12))

import shutil
sys.path.remove(str(work))
del sys.modules['fastmath']
shutil.rmtree(work)


build exit code: 0
fastmath.square(12) -> 144


### Item 97 — Rely on Precompiled Bytecode and File System Caching to Improve Startup Time

Python automatically compiles each imported `.py` file to bytecode and caches it in `__pycache__` — the *second* time a module is imported (in a later process), Python skips recompilation and loads the cached `.pyc` directly, which measurably speeds up program startup for larger codebases. This is automatic, but knowing it exists explains why the first run after a code change is often slightly slower than subsequent ones.


In [6]:
import py_compile, time
from pathlib import Path
import importlib, sys

Path('cache_demo_mod.py').write_text('x = sum(range(10000))\n')

# First compilation: creates the .pyc bytecode cache
py_compile.compile('cache_demo_mod.py', cfile='cache_demo_mod.pyc')
print('__pycache__-style bytecode cache created:', Path('cache_demo_mod.pyc').exists())

# Import timing: first import compiles (if no cache), subsequent ones reuse the cached bytecode
import cache_demo_mod
print('module loaded, x =', cache_demo_mod.x)

import os
os.remove('cache_demo_mod.py')
os.remove('cache_demo_mod.pyc')
if 'cache_demo_mod' in sys.modules:
    del sys.modules['cache_demo_mod']


__pycache__-style bytecode cache created: True
module loaded, x = 49995000


### Item 98 — Lazy-Load Modules with Dynamic Imports to Reduce Startup Time

Importing every module a program *might* need at the top of a file means paying that import cost even if a given run never uses most of them. Moving an `import` inside the function that actually needs it (a "lazy" or "dynamic" import) defers that cost until — and unless — that code path actually runs, which can meaningfully speed up startup for programs with many optional/rarely-used dependencies.


In [7]:
import time

def process_with_eager_import():
    import json   # imported every time this module loads, even if never called
    return json.dumps({'a': 1})

def process_with_lazy_import(need_json):
    if need_json:
        import json   # only paid for if this branch actually runs
        return json.dumps({'a': 1})
    return None

print(process_with_eager_import())
print(process_with_lazy_import(True))
print(process_with_lazy_import(False))   # this call never even imports json at all
print('Lazy imports matter most for large/optional dependencies rarely used on the common path.')


{"a": 1}
{"a": 1}
None
Lazy imports matter most for large/optional dependencies rarely used on the common path.


### Item 99 — Consider `memoryview` and `bytearray` for Zero-Copy Interactions with `bytes`

Slicing a plain `bytes` object (`data[1000:2000]`) **copies** that chunk into a new object — wasteful for large binary data processed in pieces. `memoryview(data)` instead gives you a view into the *same* underlying memory with no copy; slicing a memoryview is also copy-free. `bytearray` is `bytes`'s mutable counterpart, letting you modify binary data in place instead of building new `bytes` objects repeatedly.


In [8]:
data = bytes(range(256)) * 1000   # a sizable chunk of binary data

# Slicing bytes COPIES the data into a new object
sliced_copy = data[1000:2000]
print('bytes slice is a copy:', sliced_copy is not data)

# memoryview: a view into the SAME memory, no copy made
view = memoryview(data)
sliced_view = view[1000:2000]
print('memoryview slice shares the underlying buffer -- no copy of the 256000-byte data made')
print(bytes(sliced_view) == sliced_copy)   # same logical content, very different cost to produce

# bytearray: mutable, so you can modify bytes in place instead of building new objects
buffer = bytearray(10)
buffer[0] = 65
buffer[1:3] = b'BC'
print(buffer)


bytes slice is a copy: True
memoryview slice shares the underlying buffer -- no copy of the 256000-byte data made
True
bytearray(b'ABC\x00\x00\x00\x00\x00\x00\x00')
